# Audit kebocoran dan inferensi

Notebook **pelapor** untuk empat berkas audit yang tidak bernomor `expNN` tetapi mengisi
sebagian tabel terpenting disertasi:

| Skrip | Mengisi |
|---|---|
| `tools/icoict_leak_audit.py`, `tools/icoict_paper_config.py` | Tabel 4.1, Gambar 4.1 |
| `tools/leak_size_audit.py` | Tabel 4.6 |
| `tools/crossfit_audit_rossmann.py`, `tools/crossfit_audit_pharma.py` | Subbab 4.4.4 |
| `tools/groupB_inference.py` | Tabel 4.9, Tabel 4.10, Gambar 4.4 |
| `tools/rerun_lightgbm_baseline.py` | Tabel 4.7 |

Hanya membaca. Tidak melatih model, tidak menulis ke `results/`, tidak mengimpor
`src/experiments` maupun `tools`.

## 1. Ringkasan

Keempat audit ini menjawab pertanyaan yang sama dari sudut berbeda: **apakah angka yang
dilaporkan bersih dari kebocoran informasi, dan apakah kesimpulannya bertahan di bawah
inferensi yang jujur?**

| Audit | Pertanyaan | Jawaban singkat |
|---|---|---|
| ICoICT | Seberapa besar kebocoran pada studi awal penulis sendiri? | RMSE *hybrid* naik 6,8–21,2 kali (median 8,9) setelah dibetulkan |
| Ukuran kebocoran | Seberapa kecil residual *in-sample* dibanding yang jujur? | 0,62% pada S₁ linier; 5,59% dan 6,25% pada S₁ struktural |
| Audit *cross-fitted* | Apakah perbaikan R1-1 mengubah kesimpulan? | Tidak; pengaruhnya kecil dan tidak searah |
| Inferensi Grup B | Apakah kemenangan bertahan setelah dependensi dan multiplisitas? | Ya pada Rossmann; jauh lebih tipis pada data obat |

**Dipakai di mana.** Disertasi Subbab 4.1, 4.4.4, 4.5.3, 4.6.1 dan 4.7.1;
paper IJIES Tabel 4, 5, 10 dan Subbab 2.3.

## 2. Lingkungan

In [1]:
# Notebook pelapor: HANYA MEMBACA.
# Tidak melatih model, tidak mengimpor src/experiments atau tools (agar TensorFlow
# tidak ikut termuat), dan tidak pernah menulis ke results/, data/ atau raw/.
%matplotlib inline
from pathlib import Path
import ast, json, hashlib, datetime, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def cari_root(p: Path) -> Path:
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "src" / "experiments").is_dir():
            return c
    raise RuntimeError("root repositori tidak ditemukan dari " + str(p))


ROOT = cari_root(Path.cwd().resolve())
RES = ROOT / "results"

pd.set_option("display.width", 210)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_rows", 200)

print("Root repositori :", ROOT)
print("Folder hasil    :", RES.relative_to(ROOT),
      f"({sum(1 for f in RES.rglob('*') if f.is_file())} berkas)")
print("Dijalankan      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))


def idn(x, d=2, tanda=False):
    """Format angka gaya Indonesia (koma desimal, titik ribuan)."""
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    if tanda and x > 0:
        s = "+" + s
    return s.replace("-", "\u2212")


def baca(nama, **kw):
    """Baca satu berkas di results/ sebagai DataFrame. Read-only."""
    f = RES / nama
    if not f.exists():
        raise FileNotFoundError(f"{nama} tidak ada di results/")
    return pd.read_csv(f, **kw)


def potongan(relpath, nama=None, awal=None, akhir=None):
    """Tampilkan potongan kode dengan membaca berkas sebagai TEKS.

    Memakai ``ast`` untuk menemukan batas fungsi/konstanta, sehingga modul
    eksperimen tidak perlu diimpor.
    """
    f = ROOT / relpath
    teks = f.read_text(encoding="utf-8")
    baris = teks.splitlines()
    if nama is not None:
        pohon = ast.parse(teks)
        ketemu = False
        for n in ast.walk(pohon):
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.name == nama:
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
            if isinstance(n, ast.Assign) and any(getattr(t, "id", None) == nama for t in n.targets):
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
        if not ketemu:
            raise KeyError(f"'{nama}' tidak ditemukan di {relpath}")
    awal = awal or 1
    akhir = akhir or len(baris)
    lebar = len(str(akhir))
    print(f"--- {relpath}:{awal}-{akhir} " + "-" * max(0, 74 - len(relpath)))
    for i in range(awal, akhir + 1):
        print(f"{str(i).rjust(lebar)} | {baris[i - 1]}")


def git(*args):
    """Jalankan perintah git read-only di root repositori."""
    try:
        r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""


def asal(nama_meta=None, berkas_hasil=None, skrip=None, log=None):
    """Cetak asal-usul: meta.json, commit, dan log run bila ada."""
    if nama_meta:
        f = RES / nama_meta
        if f.exists():
            print("meta.json " + "-" * 60)
            print(json.dumps(json.loads(f.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
            print()
    for label, path in [("berkas hasil", berkas_hasil), ("skrip", skrip)]:
        if path:
            print(f"commit terakhir yang menyentuh {label} ({path})")
            print("  " + (git("log", "-1", "--date=short", "--format=%h  %ad  %s", "--", path) or "(tidak terbaca)"))
    if log:
        f = RES / "logs" / log
        print()
        if f.exists():
            print(f"results/logs/{log}  ({f.stat().st_size} bytes, "
                  f"ditulis {datetime.datetime.fromtimestamp(f.stat().st_mtime):%Y-%m-%d %H:%M})")
        else:
            print(f"results/logs/{log} tidak ada di klon ini (berkas *.log diabaikan git).")

Root repositori : I:\Shared drives\Riset S3\Experiment S3 Code\antgravity_riset\AR-LRX_hybrid-forecasting


Folder hasil    : results (179 berkas)
Dijalankan      : 2026-09-29 06:41


In [2]:
# Pembanding angka CSV terhadap angka yang tercetak di disertasi.
# Tidak ada angka yang dibetulkan di sini; ketidakcocokan hanya dilaporkan.
_cek = []


def cek(butir, nilai_csv, nilai_naskah, tol=0.005, d=4, rujukan=""):
    beda = abs(float(nilai_csv) - float(nilai_naskah))
    ok = beda <= tol
    _cek.append({
        "Butir": butir,
        "Rujukan naskah": rujukan,
        "Dari CSV": round(float(nilai_csv), d),
        "Di disertasi": float(nilai_naskah),
        "|selisih|": round(beda, 6),
        "Status": "COCOK" if ok else "TIDAK COCOK",
    })
    return ok


def laporan_cek():
    t = pd.DataFrame(_cek)
    n_ok = int((t.Status == "COCOK").sum())
    display(t)
    print()
    print(f"RINGKASAN: {n_ok} dari {len(t)} angka COCOK dengan naskah disertasi.")
    buruk = t[t.Status == "TIDAK COCOK"]
    if len(buruk):
        print()
        print("!" * 78)
        print("ADA ANGKA YANG TIDAK COCOK. Tidak dibetulkan di notebook ini.")
        print("Laporkan butir berikut sebelum notebook dipakai sebagai lampiran:")
        for _, r in buruk.iterrows():
            print(f"  - {r.Butir} ({r['Rujukan naskah']}): "
                  f"CSV {r['Dari CSV']} vs naskah {r['Di disertasi']}")
        print("!" * 78)
    else:
        print("Tidak ada selisih di luar toleransi pembulatan.")
    return t

### 2.1 Asal-usul: perintah asli dan commit

Notebook ini tidak menjalankan satu pun di antaranya.

In [3]:
RUN_EXPERIMENT = False

PERINTAH = [
    "python tools/icoict_paper_config.py",
    "python tools/icoict_leak_audit.py",
    "python tools/leak_size_audit.py",
    "python tools/crossfit_audit_pharma.py",
    "python tools/crossfit_audit_rossmann.py",
    "python tools/groupB_inference.py",
    "python tools/rerun_lightgbm_baseline.py",
]

if RUN_EXPERIMENT:
    for c in PERINTAH:
        print(">>", c)
        r = subprocess.run(c, shell=True, cwd=ROOT)
        print("   kode keluar:", r.returncode)
else:
    print("RUN_EXPERIMENT = False - eksperimen TIDAK dijalankan ulang.")
    print()
    print("Perintah asli yang menghasilkan berkas di results/:")
    for c in PERINTAH:
        print("   ", c)
    print()
    print('groupB_inference.py berjalan di bawah 1 menit karena hanya membaca prediksi tersimpan. crossfit_audit_pharma.py sekitar 10 menit. Sisanya beberapa menit.')

RUN_EXPERIMENT = False - eksperimen TIDAK dijalankan ulang.

Perintah asli yang menghasilkan berkas di results/:
    python tools/icoict_paper_config.py
    python tools/icoict_leak_audit.py
    python tools/leak_size_audit.py
    python tools/crossfit_audit_pharma.py
    python tools/crossfit_audit_rossmann.py
    python tools/groupB_inference.py
    python tools/rerun_lightgbm_baseline.py

groupB_inference.py berjalan di bawah 1 menit karena hanya membaca prediksi tersimpan. crossfit_audit_pharma.py sekitar 10 menit. Sisanya beberapa menit.


In [4]:
for berkas, skrip in [
        ("results/icoict_paper_config.csv", "tools/icoict_paper_config.py"),
        ("results/crossfit_leak_size.csv", "tools/leak_size_audit.py"),
        ("results/rossmann_crossfit_audit.csv", "tools/crossfit_audit_rossmann.py"),
        ("results/groupB_rossmann_dependence.csv", "tools/groupB_inference.py"),
        ("results/lightgbm_fix_audit.csv", "tools/rerun_lightgbm_baseline.py")]:
    print(f"{berkas}")
    print("   hasil : " + (git("log", "-1", "--date=short", "--format=%h  %ad  %s", "--", berkas) or "-"))
    print("   skrip : " + (git("log", "-1", "--date=short", "--format=%h  %ad  %s", "--", skrip) or "-"))

results/icoict_paper_config.csv


   hasil : e4c061d  2026-09-25  Add exp11 (calendar-keyed first stage on PharmaSales) and ICoICT leakage audit


   skrip : e4c061d  2026-09-25  Add exp11 (calendar-keyed first stage on PharmaSales) and ICoICT leakage audit
results/crossfit_leak_size.csv


   hasil : 6804e9e  2026-09-18  Update README and REPRODUCE to the cross-fitted results; add the residual-leak audit


   skrip : 6804e9e  2026-09-18  Update README and REPRODUCE to the cross-fitted results; add the residual-leak audit
results/rossmann_crossfit_audit.csv


...

   skrip : 17b43e7  2026-09-15  Prepare the cross-fitted residual rerun (R1-1): guards, exp06b reuse, audits, docs
results/groupB_rossmann_dependence.csv


   hasil : d48a0c8  2026-09-17  Enable LightGBM row bagging in the baseline and re-run dependent analyses; add exp08 and helper scripts


   skrip : d20989b  2026-09-16  Dependence-aware DM, bootstrap intervals and BH correction (R1-4, R2-5)
results/lightgbm_fix_audit.csv


   hasil : d48a0c8  2026-09-17  Enable LightGBM row bagging in the baseline and re-run dependent analyses; add exp08 and helper scripts


   skrip : d48a0c8  2026-09-17  Enable LightGBM row bagging in the baseline and re-run dependent analyses; add exp08 and helper scripts


## 3. Audit studi awal ICoICT — Tabel 4.1

**Cacatnya.** Notebook studi awal membentuk `rolling_mean_k = y.rolling(k).mean()`, dengan
jendela yang **memuat $y_t$ itu sendiri**. Nilai yang hendak diramal ikut menjadi masukan.

Dua varian dijalankan: **A** notebook apa adanya (bocor), dan **B** sama persis tetapi
rata-rata bergulir digeser satu langkah sehingga jendelanya berakhir di $t-1$. Hanya cacat
itu yang diubah, sehingga selisihnya dapat diatribusikan.

In [5]:
ic = baca("icoict_paper_config.csv")
il = baca("icoict_leak_audit.csv")

rows, rasio, hyb_vs_lr = [], [], []
for c in ic.category.unique():
    a = ic[(ic.category == c) & (ic.variant == "leaky")].iloc[0]
    b = ic[(ic.category == c) & (ic.variant == "shifted")].iloc[0]
    r = b.rmse_hybrid_avg / a.rmse_hybrid_avg
    rasio.append(r)
    hyb_vs_lr.append(100 * (b.rmse_hybrid_avg / b.rmse_LR - 1))
    rows.append([c, a.paper_rmse, round(a.rmse_hybrid_avg, 2), round(b.rmse_hybrid_avg, 2),
                 f"{r:.1f}x", round(b.rmse_LR, 2), round(b.rmse_naive, 2)])
t41 = pd.DataFrame(rows, columns=["Kategori", "Dilaporkan", "Hybrid, fitur bocor",
                                  "Hybrid, fitur diperbaiki", "Rasio",
                                  "LR, fitur diperbaiki", "Naive"])
display(t41)
print()
print(f"RMSE hybrid setelah fitur diperbaiki : {idn(ic[ic.variant=='shifted'].rmse_hybrid_avg.min(),2)}"
      f" sampai {idn(ic[ic.variant=='shifted'].rmse_hybrid_avg.max(),2)}")
print(f"Rasio perbaikan/bocor                : {idn(min(rasio),1)} sampai {idn(max(rasio),1)} kali "
      f"(median {idn(float(np.median(rasio)),1)} kali)")
print(f"Hybrid terhadap regresi linier       : rata-rata {idn(float(np.mean(hyb_vs_lr)),2,True)}%, "
      f"lebih baik pada {sum(1 for x in hyb_vs_lr if x < 0)} dari 8 kategori")
print()
print("RMSE regresi linier pada varian bocor (seharusnya nol bila bocor sempurna):")
print(f"  maksimum {ic[ic.variant=='leaky'].rmse_LR.max():.2e}")

,Kategori,Dilaporkan,"Hybrid, fitur bocor","Hybrid, fitur diperbaiki",Rasio,"LR, fitur diperbaiki",Naive
0,M01AB,0.57,0.31,2.84,9.1x,2.82,4.09
1,M01AE,0.47,0.31,2.15,6.8x,2.16,2.84
2,N02BA,0.68,0.18,2.05,11.5x,2.04,2.75
3,N02BE,1.30,1.80,12.66,7.0x,12.58,16.19
4,N05B,0.79,0.27,4.31,16.0x,4.25,5.61
5,N05C,0.54,0.05,1.12,21.2x,1.12,1.63
6,R03,1.79,1.02,8.21,8.1x,8.10,11.12
7,R06,0.03,0.25,2.18,8.7x,2.15,2.88



RMSE hybrid setelah fitur diperbaiki : 1,12 sampai 12,66
Rasio perbaikan/bocor                : 6,8 sampai 21,2 kali (median 8,9 kali)
Hybrid terhadap regresi linier       : rata-rata +0,67%, lebih baik pada 2 dari 8 kategori

RMSE regresi linier pada varian bocor (seharusnya nol bila bocor sempurna):
  maksimum 1.67e-13


Mekanisme kebocorannya, dibaca sebagai teks dari skrip auditnya:

In [6]:
# Mekanisme kebocorannya, dibaca sebagai teks dari skrip auditnya.
potongan("tools/icoict_leak_audit.py", awal=1, akhir=9)
print()
print("Varian pada icoict_leak_audit.csv (notebook asli memakai 60/20/20, paper menyatakan 80/20):")
print(il.groupby(["granularity", "split", "variant"]).size().to_string())

--- tools/icoict_leak_audit.py:1-9 ------------------------------------------------
1 | """Audit of the ICoICT 2025 pipeline (pharma-sales-daily.ipynb / pharma-sales-weekly.ipynb).
2 | Reproduces the notebook feature construction and the averaging hybrid, then removes one
3 | defect at a time:
4 |   A  notebook as written: rolling_mean_k = y.rolling(k).mean()  (window INCLUDES y_t)
5 |   B  same, but rolling mean shifted by one step (window ends at t-1)
6 | Split 80/20 as stated in the ICoICT paper (daily notebook itself used 60/20/20; both reported).
7 | Lag k = argmax ACF(1..26) on the FULL series, as in the notebook (kept in A and B so only the
8 | rolling-mean defect changes)."""
9 | import sys, numpy as np, pandas as pd, xgboost as xgb

Varian pada icoict_leak_audit.csv (notebook asli memakai 60/20/20, paper menyatakan 80/20):
granularity  split  variant  
daily        80/20  A_leaky      8
                    B_shifted    8
weekly       80/20  A_leaky      8
                    B

## 4. Seberapa besar kebocoran residual *in-sample*? — Tabel 4.6

Revisi R1-1 mengganti residual tahap pertama *in-sample* dengan residual *cross-fitted*.
Skrip ini mengukur besar selisihnya **tanpa melatih tahap kedua sama sekali**.

In [7]:
potongan("tools/leak_size_audit.py", awal=1, akhir=22)

--- tools/leak_size_audit.py:1-22 --------------------------------------------------
 1 | """
 2 | How large was the in-sample residual leak? (reviewer point R1-1, round 2)
 3 | 
 4 | The revision replaces in-sample first-stage residuals with cross-fitted ones. This
 5 | script quantifies the difference the reviewer was worried about, on the fitting block
 6 | itself, without training any second stage:
 7 | 
 8 |   * RMS of the IN-SAMPLE residual   y - S1(X)          with S1 fitted on the same rows;
 9 |   * RMS of the CROSS-FITTED residual y - S1_cf(X)      with S1 fitted on the other four
10 |     of five contiguous chronological folds;
11 |   * RMS of the residual on the VALIDATION block, i.e. what an honest out-of-block
12 |     residual looks like for the same first stage.
13 | 
14 | A first stage that memorises its own rows has an in-sample residual that is too small;
15 | the gap between the first two numbers is the size of the leak, and the third says how
16 | much of that gap 

In [8]:
lk = baca("crossfit_leak_size.csv").set_index("stage1")
t46 = pd.DataFrame({
    "Tahap pertama": lk.index,
    "In-sample": lk.rms_resid_in_sample.round(4).values,
    "Cross-fitted": lk.rms_resid_cross_fitted.round(4).values,
    "Data validasi": lk.rms_resid_validation.round(4).values,
    "Kebocoran (%)": lk.leak_pct.round(2).values,
})
display(t46)
print()
print(f"n latih = {lk.n_train.iloc[0]:,}   n validasi = {lk.n_val.iloc[0]:,}   "
      f"lipatan = {lk.n_folds.iloc[0]}")
print()
print("Kebocoran = 100 x (RMS cross-fitted / RMS in-sample - 1): seberapa jauh residual")
print("in-sample terlalu kecil dibanding residual yang dibentuk secara jujur.")
print()
print("Tahap pertama linier nyaris tidak bocor karena regresi dengan sedikit koefisien")
print("tidak dapat menghafal ratusan ribu observasi. Tahap pertama struktural bocor jauh")
print("lebih besar karena setiap observasi ikut membentuk rata-rata kelompok yang")
print("memprediksi dirinya sendiri.")

,Tahap pertama,In-sample,Cross-fitted,Data validasi,Kebocoran (%)
0,linear,0.2564,0.2580,0.2511,0.62
1,structural,0.1547,0.1634,0.2022,5.59
2,struct_linear,0.1532,0.1628,0.1987,6.25



n latih = 589,839   n validasi = 119,588   lipatan = 5

Kebocoran = 100 x (RMS cross-fitted / RMS in-sample - 1): seberapa jauh residual
in-sample terlalu kecil dibanding residual yang dibentuk secara jujur.

Tahap pertama linier nyaris tidak bocor karena regresi dengan sedikit koefisien
tidak dapat menghafal ratusan ribu observasi. Tahap pertama struktural bocor jauh
lebih besar karena setiap observasi ikut membentuk rata-rata kelompok yang
memprediksi dirinya sendiri.


## 5. Audit sebelum/sesudah residual *cross-fitted*

### 5.1 Rossmann

Kedua sisi dijalankan pada **satu lingkungan yang sama**. Ini penting: XGBoost 3.2.0 dengan
`subsample`/`colsample` < 1 tidak memberi hasil identik bit-per-bit di Windows dan Linux
(terverifikasi pada exp05c: berbeda pada digit signifikan keempat). Memasangkan "sesudah"
dari Linux dengan arsip "sebelum" dari Windows akan mencampurkan efek platform ke dalam
efek R1-1.

In [9]:
cfa = baca("rossmann_crossfit_audit.csv")
kol = ["model", "orig_RMSE_archived", "orig_RMSE_before", "orig_RMSE_after",
       "orig_RMSE_change_pct", "legacy_reproduces_archived"]
display(cfa[kol].round(4))
print()
n_ok = int(cfa.legacy_reproduces_archived.astype(str).str.lower().eq("true").sum())
print(f"Mode legacy mereproduksi angka arsip: {n_ok} dari {len(cfa)} model.")
print()
if n_ok == len(cfa):
    print("INI CEK YANG PALING PENTING. Kalau mode legacy (cross_fit_folds=None) tidak")
    print("mereproduksi angka yang diarsipkan, berarti ada hal LAIN yang berubah selain")
    print("prosedur residual, dan perbandingan sebelum/sesudah menjadi tidak sah.")
else:
    print("!!! ADA MODEL YANG TIDAK MEREPRODUKSI ARSIP. Laporkan.")
print()
print(f"Platform audit : {cfa.platform.iloc[0]}, Python {cfa.python.iloc[0]}, "
      f"{int(cfa.cross_fit_folds.iloc[0])} lipatan")
print()
print("Pengaruh terhadap akurasi uji kecil dan TIDAK SEARAH:")
for _, r in cfa.iterrows():
    print(f"  {r.model:38s} {idn(r.orig_RMSE_before,2)} -> {idn(r.orig_RMSE_after,2)}  "
          f"({idn(r.orig_RMSE_change_pct,2,True)}%)")

,model,orig_RMSE_archived,orig_RMSE_before,orig_RMSE_after,orig_RMSE_change_pct,legacy_reproduces_archived
0,AR-LRX [struct_linear],1004.6948,1004.6948,998.1492,-0.6515,True
1,"LR-XGB (residual, tanpa gerbang)",1058.8243,1058.8243,1061.4973,0.2524,True
2,AR-LRX-Aug [structural],945.4213,945.4213,954.9046,1.0031,True
3,AR-LRX-Aug [struct_linear],956.2672,956.2672,961.3417,0.5307,True



Mode legacy mereproduksi angka arsip: 4 dari 4 model.

INI CEK YANG PALING PENTING. Kalau mode legacy (cross_fit_folds=None) tidak
mereproduksi angka yang diarsipkan, berarti ada hal LAIN yang berubah selain
prosedur residual, dan perbandingan sebelum/sesudah menjadi tidak sah.

Platform audit : Windows-10-10.0.22621-SP0, Python 3.10.11, 5 lipatan

Pengaruh terhadap akurasi uji kecil dan TIDAK SEARAH:
  AR-LRX [struct_linear]                 1.004,69 -> 998,15  (−0,65%)
  LR-XGB (residual, tanpa gerbang)       1.058,82 -> 1.061,50  (+0,25%)
  AR-LRX-Aug [structural]                945,42 -> 954,90  (+1,00%)
  AR-LRX-Aug [struct_linear]             956,27 -> 961,34  (+0,53%)


### 5.2 PharmaSales

In [10]:
cs = baca("exp05c_crossfit_audit_summary.csv", index_col=0)
display(cs.round(4))
print()
print("Kolom 'legacy' = prosedur naskah yang dikirim (residual in-sample);")
print("kolom 'crossfit' = prosedur revisi. Baris T7-T9 merujuk tabel paper IJIES.")

,legacy,crossfit,change
"T7 gate closed (w*=0), of N",15.0000,17.0000,2.0000
"T7 gate closed, daily",7.0000,6.0000,-1.0000
"T7 gate closed, weekly",8.0000,11.0000,3.0000
T7 mean selected w*,0.1531,0.1375,-0.0156
T7 count w*=0,15.0000,17.0000,2.0000
T7 count w*=0.1,7.0000,5.0000,-2.0000
T7 count w*=0.2,4.0000,5.0000,1.0000
T7 count w*=0.5,2.0000,NaN,NaN
T7 count w*=0.6,4.0000,NaN,NaN
"resid_val_r2 negative, of N",30.0000,31.0000,1.0000



Kolom 'legacy' = prosedur naskah yang dikirim (residual in-sample);
kolom 'crossfit' = prosedur revisi. Baris T7-T9 merujuk tabel paper IJIES.


## 6. Inferensi yang sadar dependensi dan multiplisitas

`groupB_inference.py` **tidak melatih apa pun**. Ia membaca prediksi uji yang tersimpan
(`results/*_predictions.npz`, tidak dilacak git) dan menulis ringkasan.

### 6.1 Rossmann — Tabel 4.9

Uji DM pada Tabel 4 paper memperlakukan 129.333 selisih rugi toko-hari sebagai saling bebas.
Di sini model yang diusulkan dibandingkan dengan setiap pembanding di bawah empat prosedur:
DM tingkat observasi (acuan saja), DM atas rata-rata per tanggal dengan ragam Newey–West,
dan *bootstrap* blok per tanggal, klaster toko, serta dua arah.

In [11]:
potongan("tools/groupB_inference.py", nama="newey_west_dm")

--- tools/groupB_inference.py:105-117 -------------------------------------------------
105 | def newey_west_dm(d: np.ndarray, lag: int):
106 |     """DM statistic with a Bartlett-kernel long-run variance; t(T-1) reference."""
107 |     d = np.asarray(d, dtype=float)
108 |     T = len(d)
109 |     dbar = d.mean()
110 |     u = d - dbar
111 |     lrv = u @ u / T
112 |     for k in range(1, lag + 1):
113 |         lrv += 2 * (1 - k / (lag + 1)) * (u[k:] @ u[:-k]) / T
114 |     if lrv <= 0:
115 |         return np.nan, np.nan
116 |     stat = dbar / np.sqrt(lrv / T)
117 |     return float(stat), float(2 * stats.t.sf(abs(stat), df=T - 1))


In [12]:
dep = baca("groupB_rossmann_dependence.csv")
d = dep[dep.scale == "orig"].set_index("reference")
URUT = [("LightGBM (Zeng)", "LightGBM"), ("XGBoost (grid Zhao)", "XGBoost (grid Zhao)"),
        ("XGBoost", "XGBoost, grid protokol"),
        ("LR-XGB (residual, tanpa gerbang)", "Hybrid residual tanpa gerbang"),
        ("AR-LRX [struct_linear]", "AR-LRX bergerbang, S1 struct_linear"),
        ("S1 [structural]", "S1 struktural"),
        ("MLP (Diamantini)", "MLP"), ("CNN (Diamantini)", "CNN"), ("RNN (Diamantini)", "RNN"),
        ("LSTM (Diamantini / Qureshi)", "LSTM"), ("GRU (Qureshi)", "GRU"),
        ("Transformer (Diamantini)", "Transformer")]
rows = []
for k, lab in URUT:
    if k not in d.index:
        continue
    x = d.loc[k]
    rows.append([lab, round(x.rel_diff_pct, 2), round(x.obs_DM, 1), round(x.dateDM_NW4, 2),
                 "< 0,001" if x.dateDM_NW4_p < 0.001 else round(x.dateDM_NW4_p, 3),
                 f"[{x.twoway_ci_lo_pct:.1f}; {x.twoway_ci_hi_pct:.1f}]"])
t49 = pd.DataFrame(rows, columns=["Pembanding", "Delta (%)", "DM observasi",
                                  "DM per tanggal", "p", "SK 95% (%)"])
display(t49)
print()
print(f"n observasi = {int(d.n_obs.iloc[0]):,}   n tanggal = {int(d.n_dates.iloc[0]):,}   "
      f"n toko = {int(d.n_stores.iloc[0]):,}")
print()
print("Perbandingan DM observasi dengan DM per tanggal memperlihatkan mengapa koreksi")
print("dependensi penting: statistik tingkat observasi jauh lebih besar karena menganggap")
print("ratusan ribu selisih rugi sebagai saling bebas.")

,Pembanding,Delta (%),DM observasi,DM per tanggal,p,SK 95% (%)
0,LightGBM,-5.45,-20.6,-3.12,0.002,[-9.6; -1.6]
1,XGBoost (grid Zhao),-5.85,-19.3,-4.03,"< 0,001",[-9.6; -2.4]
2,"XGBoost, grid protokol",-6.18,-21.0,-3.96,"< 0,001",[-10.2; -2.5]
3,Hybrid residual tanpa gerbang,-10.04,-16.4,-5.70,"< 0,001",[-16.2; -5.1]
4,"AR-LRX bergerbang, S1 struct_linear",-4.33,-19.1,-3.36,0.001,[-6.5; -2.0]
5,S1 struktural,-23.03,-64.7,-3.95,"< 0,001",[-28.4; -17.6]
6,MLP,-24.05,-52.4,-9.04,"< 0,001",[-29.3; -18.9]
7,CNN,-25.88,-54.0,-9.20,"< 0,001",[-31.1; -21.1]
8,RNN,-23.16,-44.7,-8.46,"< 0,001",[-28.5; -17.9]
9,LSTM,-23.84,-56.3,-8.45,"< 0,001",[-28.8; -18.9]



n observasi = 129,333   n tanggal = 141   n toko = 1,115

Perbandingan DM observasi dengan DM per tanggal memperlihatkan mengapa koreksi
dependensi penting: statistik tingkat observasi jauh lebih besar karena menganggap
ratusan ribu selisih rugi sebagai saling bebas.


### 6.2 PharmaSales — Tabel 4.10

Koreksi Benjamini–Hochberg dan selang *bootstrap* per konfigurasi, dengan penggabungan yang
diklaster per kategori.

In [13]:
gj = json.loads((RES / "groupB_summary.json").read_text(encoding="utf-8"))
pooled = baca("groupB_pharma_pooled.csv")

print(f"Bootstrap B = {gj['B']}, seed {gj['seed']}")
print(f"Uji valid PharmaSales        : {gj['pharma_valid_tests']} "
      f"(tak terdefinisi: {gj['pharma_undefined_tests']})")
print(f"Konfigurasi duplikat         : {gj['pharma_duplicate_configs']}")
print(f"Uji valid pada 20 konfigurasi: {gj['pharma_valid_tests_distinct']}")
print()
rows = []
for k in ["Naive", "XGBoost", "LR-XGB (residual, tanpa gerbang)", "S1 [linear]"]:
    g = gj[k]
    p = pooled[(pooled.reference == k) & (pooled.subset == "all32") & (pooled.granularity == "all")].iloc[0]
    rows.append([k, p.configs_better, p.configs_equal, p.configs_worse,
                 f"{g['sig_win_raw']} ({g['sig_win_BH_all']})",
                 f"{g['sig_loss_raw']} ({g['sig_loss_BH_all']})",
                 round(p.geo_mean_rel_diff_pct, 2),
                 f"[{p.ci_lo_pct:.2f}; {p.ci_hi_pct:.2f}]"])
display(pd.DataFrame(rows, columns=["Pembanding", "Menang", "Seri", "Kalah",
                                    "Menang sig. (BH)", "Kalah sig. (BH)",
                                    "Gabungan (%)", "SK 95% (%)"]))
print()
print("Ini Tabel 4.10 disertasi. Angka dalam kurung adalah jumlah setelah koreksi")
print("Benjamini-Hochberg: 20 dari 22 kemenangan atas naive bertahan, tetapi hanya")
print("2 dari 4 atas XGBoost dan 2 dari 5 atas hybrid tanpa gerbang.")

Bootstrap B = 2000, seed 42
Uji valid PharmaSales        : 111 (tak terdefinisi: 17)
Konfigurasi duplikat         : 12
Uji valid pada 20 konfigurasi: 69



,Pembanding,Menang,Seri,Kalah,Menang sig. (BH),Kalah sig. (BH),Gabungan (%),SK 95% (%)
0,Naive,30,0,2,22 (20),0 (0),-16.69,[-21.27; -11.81]
1,XGBoost,25,0,7,4 (2),0 (0),-1.63,[-3.11; 0.04]
2,"LR-XGB (residual, tanpa gerbang)",21,0,11,5 (2),0 (0),-1.45,[-3.17; 0.27]
3,S1 [linear],7,17,8,0 (0),2 (0),0.21,[-0.05; 0.54]



Ini Tabel 4.10 disertasi. Angka dalam kurung adalah jumlah setelah koreksi
Benjamini-Hochberg: 20 dari 22 kemenangan atas naive bertahan, tetapi hanya
2 dari 4 atas XGBoost dan 2 dari 5 atas hybrid tanpa gerbang.


## 7. Perbaikan baseline LightGBM — Tabel 4.7

In [14]:
lg = baca("lightgbm_fix_audit.csv")
display(lg.round(4))
print()
print("Cacat yang diperbaiki: baselines.fp_lightgbm melewatkan subsample=0.8 TANPA")
print("subsample_freq. LightGBM mengabaikan subsampling baris selama frekuensinya 0,")
print("sehingga baseline LightGBM yang dikirim berjalan TANPA row bagging.")
print()
print("Skrip perbaikannya lebih dahulu memeriksa bahwa setelan lama dapat direproduksi,")
print("baru mengganti HANYA baris LightGBM pada berkas hasil exp06b.")

,model,platform,orig_RMSE_archived,orig_MAE_archived,orig_RMSPE_archived,orig_R2_archived,test_RMSE_archived,val_RMSE_archived,val_R2_archived,orig_RMSE_freq0,orig_MAE_freq0,orig_RMSPE_freq0,orig_R2_freq0,test_RMSE_freq0,val_RMSE_freq0,val_R2_freq0,orig_RMSE_freq1,orig_MAE_freq1,orig_RMSPE_freq1,orig_R2_freq1,test_RMSE_freq1,val_RMSE_freq1,val_R2_freq1,params_freq0,params_freq1,orig_RMSE_change_pct
0,LightGBM (Zeng),Windows-10-10.0.22621-SP0,1015.7271,697.1374,0.1324,0.8943,0.1307,0.1853,0.808,1015.7271,697.1374,0.1324,0.8943,0.1307,0.1853,0.808,1009.9881,692.4868,0.1318,0.8955,0.1299,0.1848,0.809,"{""colsample_bytree"": 0.8, ""learning_rate"": 0.1, ""n_estimators"": 900, ""num_leaves"": 127, ""su...","{""colsample_bytree"": 0.8, ""learning_rate"": 0.1, ""n_estimators"": 900, ""num_leaves"": 127, ""su...",-0.565



Cacat yang diperbaiki: baselines.fp_lightgbm melewatkan subsample=0.8 TANPA
subsample_freq. LightGBM mengabaikan subsampling baris selama frekuensinya 0,
sehingga baseline LightGBM yang dikirim berjalan TANPA row bagging.

Skrip perbaikannya lebih dahulu memeriksa bahwa setelan lama dapat direproduksi,
baru mengganti HANYA baris LightGBM pada berkas hasil exp06b.


## 8. Cek kecocokan dengan angka disertasi

In [15]:
# --- Tabel 4.6 / Subbab 4.4.4 -----------------------------------------------
cek("Tabel 4.6: kebocoran S1 linier (%)", lk.loc["linear", "leak_pct"], 0.62,
    tol=0.005, rujukan="Tabel 4.6 / Subbab 4.4.4")
cek("Tabel 4.6: kebocoran S1 struktural (%)", lk.loc["structural", "leak_pct"], 5.59,
    tol=0.005, rujukan="Tabel 4.6 / Subbab 4.4.4")
cek("Tabel 4.6: kebocoran S1 struct_linear (%)", lk.loc["struct_linear", "leak_pct"], 6.25,
    tol=0.005, rujukan="Tabel 4.6 / Subbab 4.4.4")
cek("Subbab 4.4.4: jumlah observasi latih", lk.n_train.iloc[0], 589839, tol=0,
    rujukan="Subbab 4.4.4")

# --- Audit cross-fitted Rossmann --------------------------------------------
cek("Subbab 4.4.4: mode legacy mereproduksi arsip (seluruh model)",
    int(cfa.legacy_reproduces_archived.astype(str).str.lower().eq("true").sum()), len(cfa),
    tol=0, rujukan="Subbab 4.4.4")
cfx = cfa.set_index("model")
cek("Subbab 4.4.4: AR-LRX-Aug [structural] sebelum",
    cfx.loc["AR-LRX-Aug [structural]", "orig_RMSE_before"], 945.42, tol=0.005, rujukan="Subbab 4.4.4")
cek("Subbab 4.4.4: AR-LRX-Aug [structural] sesudah",
    cfx.loc["AR-LRX-Aug [structural]", "orig_RMSE_after"], 954.90, tol=0.005, rujukan="Subbab 4.4.4")

# --- Tabel 4.9 / Subbab 4.7.1 -----------------------------------------------
cek("Subbab 4.7.1: vs LightGBM, pembanding terkuat (%)", d.loc["LightGBM (Zeng)"].rel_diff_pct,
    -5.45, tol=0.005, rujukan="Subbab 4.7.1")
cek("Subbab 4.7.1: vs XGBoost grid protokol (%)", d.loc["XGBoost"].rel_diff_pct, -6.18,
    tol=0.005, rujukan="Subbab 4.7.1")
cek("Subbab 4.7.1: vs hybrid residual tanpa gerbang (%)",
    d.loc["LR-XGB (residual, tanpa gerbang)"].rel_diff_pct, -10.04, tol=0.005, rujukan="Subbab 4.7.1")
neural = ["MLP (Diamantini)", "CNN (Diamantini)", "RNN (Diamantini)",
          "LSTM (Diamantini / Qureshi)", "GRU (Qureshi)"]
cek("Subbab 4.7.1: vs model neural, batas atas (%)", d.loc[neural].rel_diff_pct.max(), -23.16,
    tol=0.01, rujukan="Subbab 4.7.1 (23-26%)")
cek("Subbab 4.7.1: vs model neural, batas bawah (%)", d.loc[neural].rel_diff_pct.min(), -25.88,
    tol=0.01, rujukan="Subbab 4.7.1 (23-26%)")

# --- Tabel 4.10 --------------------------------------------------------------
pv = pooled[(pooled.subset == "all32") & (pooled.granularity == "all")].set_index("reference")
for k, menang, sig_raw, sig_bh, gab in [
        ("Naive", 30, 22, 20, -16.69), ("XGBoost", 25, 4, 2, -1.63),
        ("LR-XGB (residual, tanpa gerbang)", 21, 5, 2, -1.45)]:
    cek(f"Tabel 4.10 {k}: menang", pv.loc[k].configs_better, menang, tol=0, rujukan="Tabel 4.10")
    cek(f"Tabel 4.10 {k}: menang signifikan mentah", gj[k]["sig_win_raw"], sig_raw, tol=0,
        rujukan="Tabel 4.10")
    cek(f"Tabel 4.10 {k}: menang signifikan setelah BH", gj[k]["sig_win_BH_all"], sig_bh, tol=0,
        rujukan="Tabel 4.10")
    cek(f"Tabel 4.10 {k}: gabungan (%)", pv.loc[k].geo_mean_rel_diff_pct, gab, tol=0.005,
        rujukan="Tabel 4.10 / Subbab 4.6.1")
cek("Tabel 4.10 S1 [linear]: menang", pv.loc["S1 [linear]"].configs_better, 7, tol=0, rujukan="Tabel 4.10")
cek("Tabel 4.10 S1 [linear]: seri", pv.loc["S1 [linear]"].configs_equal, 17, tol=0, rujukan="Tabel 4.10")
cek("Tabel 4.10 S1 [linear]: kalah signifikan mentah", gj["S1 [linear]"]["sig_loss_raw"], 2, tol=0,
    rujukan="Tabel 4.10")
cek("Tabel 4.10 S1 [linear]: kalah signifikan setelah BH", gj["S1 [linear]"]["sig_loss_BH_all"], 0,
    tol=0, rujukan="Tabel 4.10")
cek("Tabel 4.10 S1 [linear]: gabungan (%)", pv.loc["S1 [linear]"].geo_mean_rel_diff_pct, 0.21,
    tol=0.005, rujukan="Tabel 4.10")

# --- Subbab 4.1.3: rekonstruksi studi awal ----------------------------------
r06 = ic[(ic.category == "R06") & (ic.variant == "leaky")].iloc[0]
cek("Subbab 4.1.3: R06, hybrid fitur bocor", r06.rmse_hybrid_avg, 0.25, tol=0.005,
    rujukan="Subbab 4.1.3")
cek("Subbab 4.1.3: R06, angka yang dilaporkan studi awal", r06.paper_rmse, 0.03, tol=0.0005,
    rujukan="Subbab 4.1.3")
cek("Subbab 4.1.3: jumlah kategori ATC", ic.category.nunique(), 8, tol=0, rujukan="Tabel 4.1")

hasil_cek = laporan_cek()

,Butir,Rujukan naskah,Dari CSV,Di disertasi,|selisih|,Status
0,Tabel 4.6: kebocoran S1 linier (%),Tabel 4.6 / Subbab 4.4.4,0.6216,0.62,0.001646,COCOK
1,Tabel 4.6: kebocoran S1 struktural (%),Tabel 4.6 / Subbab 4.4.4,5.5880,5.59,0.001962,COCOK
2,Tabel 4.6: kebocoran S1 struct_linear (%),Tabel 4.6 / Subbab 4.4.4,6.2523,6.25,0.002276,COCOK
3,Subbab 4.4.4: jumlah observasi latih,Subbab 4.4.4,589839.0000,589839.00,0.000000,COCOK
4,Subbab 4.4.4: mode legacy mereproduksi arsip (seluruh model),Subbab 4.4.4,4.0000,4.00,0.000000,COCOK
5,Subbab 4.4.4: AR-LRX-Aug [structural] sebelum,Subbab 4.4.4,945.4213,945.42,0.001292,COCOK
6,Subbab 4.4.4: AR-LRX-Aug [structural] sesudah,Subbab 4.4.4,954.9046,954.90,0.004569,COCOK
7,"Subbab 4.7.1: vs LightGBM, pembanding terkuat (%)",Subbab 4.7.1,-5.4539,-5.45,0.003882,COCOK
8,Subbab 4.7.1: vs XGBoost grid protokol (%),Subbab 4.7.1,-6.1824,-6.18,0.002408,COCOK
9,Subbab 4.7.1: vs hybrid residual tanpa gerbang (%),Subbab 4.7.1,-10.0417,-10.04,0.001734,COCOK



RINGKASAN: 32 dari 32 angka COCOK dengan naskah disertasi.
Tidak ada selisih di luar toleransi pembulatan.


## 9. Interpretasi dan keterbatasan

**Audit ICoICT memeriksa kebocoran pada publikasi penulis sendiri.** Setelah fitur dibetulkan,
RMSE *hybrid* naik 6,8 sampai 21,2 kali nilai bocornya, dengan median 8,9 kali, dan
keunggulan *hybrid* atas regresi linier biasa hilang. RMSE regresi linier pada varian bocor
mendekati nol, yang menandakan nilai target ikut menjadi masukan.

**Ukuran kebocoran residual menjelaskan mengapa R1-1 perlu.** Pada tahap pertama linier
residual *in-sample* hanya 0,62% lebih kecil daripada yang *cross-fitted*, karena regresi
dengan sedikit koefisien tidak dapat menghafal 589.839 observasi. Pada tahap pertama
struktural selisihnya 5,59% dan 6,25%, karena setiap observasi ikut membentuk rata-rata
kelompok yang memprediksinya sendiri.

**Tetapi dampaknya terhadap akurasi uji kecil dan tidak searah.** Itulah temuan yang
menenangkan sekaligus harus disampaikan jujur: perbaikan prosedur yang secara prinsip benar
ternyata mengubah angka hanya sekitar satu persen, dan tidak selalu ke arah yang sama.

**Inferensi Grup B mempertipis klaim pada data obat secara nyata.** Setelah koreksi
Benjamini–Hochberg, 20 dari 22 kemenangan atas naive bertahan, tetapi hanya 2 dari 4 atas
XGBoost dan 2 dari 5 atas *hybrid* tanpa gerbang. Selang kepercayaan untuk XGBoost dan
*hybrid* tanpa gerbang **memuat nol**. Pada Rossmann, seluruh selisih tetap signifikan
bahkan di bawah DM per tanggal dengan Newey–West.

**Keterbatasan.**

1. Rekonstruksi studi awal tidak dapat menyamai angka yang dilaporkan secara persis
   (misalnya 0,25 berbanding 0,03 untuk R06), karena *tuning* hiperparameter studi awal
   dinilai terhadap data uji dan tidak dapat direkonstruksi. Arah kesimpulan tidak
   bergantung pada selisih itu.
2. `groupB_inference.py` membaca `*_predictions.npz` yang **tidak dilacak git**, sehingga
   hasilnya tidak dapat dihitung ulang dari klon baru tanpa menjalankan ulang eksperimen
   sumbernya.
3. Audit *cross-fitted* Rossmann hanya mencakup varian V3 secara bawaan.
4. *Bootstrap* memakai B = 2000; selang yang sangat sempit akan terbatas presisinya.

## 10. Pertanyaan yang mungkin diajukan promotor

**1. "Anda memeriksa kebocoran pada publikasi Anda sendiri. Bagaimana posisinya dalam
penelitian ini?"**
Ketiga cacatnya dinyatakan terbuka pada Subbab 4.1, yaitu rata-rata bergulir yang memuat
nilai target, *tuning* yang memakai RMSE data uji sebagai fungsi objektif, dan jumlah lag
yang ditentukan dari autokorelasi seluruh deret termasuk periode uji. Kontrak eksperimen
C1–C8 disusun untuk mencegah ketiganya terulang pada seluruh eksperimen berikutnya.

**2. "Mengapa reproduksi tidak menghasilkan angka yang sama dengan yang dipublikasikan?"**
Karena *tuning* pada studi awal dinilai terhadap data uji, sehingga tidak dapat
direkonstruksi secara tepat. Contohnya R06: reproduksi bocor memberi 0,25 sedangkan yang
dilaporkan 0,03. Arah kesimpulannya tidak bergantung pada selisih itu — kenaikan *error*
akibat perbaikan fitur jauh lebih besar daripada perbedaan reproduksi.

**3. "Perbaikan residual *cross-fitted* hanya mengubah angka sekitar 1%. Apakah revisinya
sepadan?"**
Sepadan, karena yang dipersoalkan bukan besar angkanya melainkan keabsahan prosedurnya.
Yang penting untuk dilaporkan adalah dua-duanya: bahwa prosedurnya kini benar, **dan** bahwa
dampaknya kecil dan tidak searah. Cek "mode legacy mereproduksi arsip" pada bagian 5.1
memastikan tidak ada hal lain yang ikut berubah.

**4. "Apakah kemenangan pada data obat bertahan setelah koreksi multiplisitas?"**
Sebagian besar tidak, dan itu dilaporkan. Atas naive bertahan (20 dari 22), tetapi atas
XGBoost hanya 2 dari 4 dan atas *hybrid* tanpa gerbang 2 dari 5, dengan selang kepercayaan
yang memuat nol. Disertasi karena itu tidak mengklaim keunggulan yang bermakna atas XGBoost
pada data obat.

**5. "Mengapa uji DM tingkat observasi tidak dipakai saja?"**
Karena ia menganggap 129.333 selisih rugi toko-hari saling bebas, padahal toko-toko berbagi
hari dan hari-hari berbagi toko. Bandingkan kolom "DM observasi" dan "DM per tanggal" pada
Tabel 4.9: statistiknya turun drastis. Yang dilaporkan sebagai bukti adalah versi per
tanggal dengan Newey–West dan selang *bootstrap* dua arah, bukan versi tingkat observasi.